# Lab 11-05 · Fourier Series

Companion to [chapter 11-05](../../chapters/11-05-fourier-series.md). Four parts: computing coefficients, the Gibbs phenomenon, Parseval and THD, and filtering a PWM signal harmonic by harmonic. Run the cells in order.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, solve_ivp

## Part 1 — Coefficients two ways

Compute $c_k = \frac{1}{T_0}\int_{T_0}x(t)e^{-jk\omega_0t}dt$ by numerical integration, and again from an FFT of one sampled period (`np.fft.fft(x) / M` gives $c_k$ when one period is sampled at $M$ points). Compare with the closed forms.

In [2]:
def coeff_quad(x, k, T0=1.0, breaks=()):
    w0 = 2 * np.pi / T0
    re = quad(lambda t: x(t) * np.cos(k * w0 * t), 0, T0, points=breaks, limit=400)[0]
    im = quad(lambda t: -x(t) * np.sin(k * w0 * t), 0, T0, points=breaks, limit=400)[0]
    return (re + 1j * im) / T0

def coeff_fft(x, K, M=4096, T0=1.0):
    t = (np.arange(M) + 0.5) * T0 / M                    # midpoints avoid sampling exactly on a jump
    X = np.fft.fft(x(t)) / M
    return X[:K + 1] * np.exp(1j * np.pi * np.arange(K + 1) / M)   # undo the half-sample shift

square = lambda t: np.where((np.asarray(t) % 1.0) < 0.5, 1.0, -1.0)
pulse = lambda t: np.where((np.asarray(t) % 1.0) < 0.25, 3.3, 0.0)
cf = coeff_fft(square, 5)
for k in range(1, 6):
    q = coeff_quad(square, k, breaks=(0.5,))
    print(f"square k={k}: quad {abs(q):.4f}, FFT {abs(cf[k]):.4f}, closed form {4 / (np.pi * k) / 2 if k % 2 else 0:.4f}")
for k in range(0, 5):
    q = coeff_quad(pulse, k, breaks=(0.25,))
    print(f"PWM k={k}: |c_k| = {abs(q):.4f} V, closed form {3.3 * 0.25 * abs(np.sinc(k * 0.25)):.4f} V")

square k=1: quad 0.6366, FFT 0.6366, closed form 0.6366
square k=2: quad 0.0000, FFT 0.0000, closed form 0.0000
square k=3: quad 0.2122, FFT 0.2122, closed form 0.2122
square k=4: quad 0.0000, FFT 0.0000, closed form 0.0000
square k=5: quad 0.1273, FFT 0.1273, closed form 0.1273
PWM k=0: |c_k| = 0.8250 V, closed form 0.8250 V
PWM k=1: |c_k| = 0.7428 V, closed form 0.7428 V
PWM k=2: |c_k| = 0.5252 V, closed form 0.5252 V
PWM k=3: |c_k| = 0.2476 V, closed form 0.2476 V
PWM k=4: |c_k| = 0.0000 V, closed form 0.0000 V


## Part 2 — The Gibbs phenomenon

Sum the square wave's series up to harmonic $N$ and find the first peak after the jump at $t = 0$.

In [3]:
def partial(theta, N):
    return sum(4 / (np.pi * k) * np.sin(k * theta) for k in range(1, N + 1, 2))

for N in (9, 99, 999):
    th = np.linspace(1e-6, 3 * np.pi / (N + 1), 30001)
    y = partial(th, N)
    i = np.argmax(y)
    print(f"N = {N:4d}: peak {y[i]:.5f} at w0 t = {th[i]:.5f} (pi/(N+1) = {np.pi / (N + 1):.5f})")

N =    9: peak 1.18233 at w0 t = 0.31416 (pi/(N+1) = 0.31416)
N =   99: peak 1.17901 at w0 t = 0.03142 (pi/(N+1) = 0.03142)
N =  999: peak 1.17898 at w0 t = 0.00314 (pi/(N+1) = 0.00314)


## Part 3 — Parseval and THD

Check $P_x = \sum\lvert c_k\rvert^2$ for the square wave and the triangle, and compute THD from the total power and the fundamental.

In [4]:
def power_from_coeffs(x, K, breaks):
    return abs(coeff_quad(x, 0, breaks=breaks)) ** 2 + 2 * sum(abs(coeff_quad(x, k, breaks=breaks)) ** 2 for k in range(1, K + 1))

triangle = lambda t: 1 - 4 * np.abs((np.asarray(t) % 1.0) - 0.5)          # peaks +1 at t = 0.5, -1 at t = 0
for name, x, br, P in (("square", square, (0.5,), 1.0), ("triangle", triangle, (0.5,), 1 / 3)):
    Pk = power_from_coeffs(x, 199, br)
    A1 = 2 * abs(coeff_quad(x, 1, breaks=br))
    thd = np.sqrt((P - A1 ** 2 / 2) / (A1 ** 2 / 2))
    print(f"{name:8s}: power {P:.4f}, sum of |c_k|^2 up to k = 199: {Pk:.4f}, THD {100 * thd:.2f}%")

square  : power 1.0000, sum of |c_k|^2 up to k = 199: 0.9980, THD 48.34%


triangle: power 0.3333, sum of |c_k|^2 up to k = 199: 0.3333, THD 12.12%


## Part 4 — Filtering a PWM signal (guided exercise)

**Predict first:** replacing the RC filter of Example 11-05.7 with a second-order filter with the same 1 kHz corner, how much smaller does the 100 kHz ripple become? Then run the cell, which computes the ripple harmonic by harmonic and checks it against a time-domain simulation of the RC circuit.

In [5]:
A, D, f0, fc = 3.3, 0.25, 100e3, 1e3
T0 = 1 / f0
t = np.linspace(0, T0, 4001)

def output(H, K=3000):
    y = np.full_like(t, A * D * H(0).real)
    for k in range(1, K + 1):
        ck = A * D * np.sinc(k * D) * np.exp(-1j * np.pi * k * D)
        y += 2 * np.real(ck * H(k * f0) * np.exp(1j * 2 * np.pi * k * f0 * t))
    return y

H1 = lambda f: 1 / (1 + 1j * f / fc)
H2 = lambda f: 1 / (1 + 1j * f / fc) ** 2                  # two isolated RC sections, same corner
y1, y2 = output(H1), output(H2)
print(f"first order : 100 kHz ripple amplitude {2 * 0.7428 * abs(H1(f0)) * 1e3:.3f} mV, peak-to-peak {(y1.max() - y1.min()) * 1e3:.2f} mV")
print(f"second order: 100 kHz ripple amplitude {2 * 0.7428 * abs(H2(f0)) * 1e3:.4f} mV, peak-to-peak {(y2.max() - y2.min()) * 1e3:.4f} mV")

RC = 1 / (2 * np.pi * fc)
src = lambda tt: A if (tt % T0) < D * T0 else 0.0
sim = solve_ivp(lambda tt, v: [(src(tt) - v[0]) / RC], (0, 2e-3), [A * D], max_step=T0 / 200, rtol=1e-9, atol=1e-12,
                dense_output=True)
tail = np.linspace(2e-3 - T0, 2e-3, 2001)
v = sim.sol(tail)[0]
print(f"time-domain RC simulation: peak-to-peak {(v.max() - v.min()) * 1e3:.2f} mV around {v.mean():.4f} V")

first order : 100 kHz ripple amplitude 14.855 mV, peak-to-peak 38.87 mV
second order: 100 kHz ripple amplitude 0.1485 mV, peak-to-peak 0.3053 mV


time-domain RC simulation: peak-to-peak 38.87 mV around 0.8250 V


**What to take away.** Coefficients can be computed by integration or by an FFT of one period, and both match the closed forms. The Gibbs overshoot sits at about 1.179 however many harmonics are used. Parseval turns power and THD into sums of coefficients. And an LTI filter acts on a periodic signal one harmonic at a time: the steady-state output computed that way matches a direct simulation of the circuit.